# Concurrent builds and leases

Many processes, possibly on many cluster nodes, may ask for the same record at once. tensordir
builds it once:

1. A process that finds no finished record tries to take the record's **lease**: an atomic
   `mkdir` of `.tensordir/locks/<type>/<key16>/`, which works on local disks, NFS and
   Lustre/GPFS (no `flock`).
2. The lease holder builds in `.tensordir/tmp/` and renames the result into place, so readers
   never see a half-written record. A heartbeat thread refreshes `owner.json` in the lease.
3. Everyone else waits with exponential backoff (capped at 2 s) and returns the finished record.

A lease whose heartbeat is older than its time to live (`lease_ttl`) is **stale**: the builder
died, and anyone may break the lease and build again. This notebook races processes for one
record, inspects a lease, kills a builder, and edits a workspace record from many processes.

In [ ]:
import json
import os
import signal
import subprocess
import sys
import tempfile
import textwrap
import time
from pathlib import Path

import tensordir as td

tmp = Path(tempfile.mkdtemp(prefix="tensordir-concurrency-"))
root = tmp / "results"
td.set_root(root)

## A record type that other processes can import

Worker processes import record classes by module, so this notebook writes them to a module
file. `Slow` sleeps during its build and logs which process built it. `Fragile` has a short
`lease_ttl` so that a dead builder's lease goes stale quickly.

In [ ]:
(tmp / "jobs.py").write_text(textwrap.dedent('''
    import os
    import time

    import numpy as np

    import tensordir as td


    class Slow(td.Record):
        __record_name__ = "Slow"

        class Spec(td.Spec):
            seconds: float = 1.0

        def build(self, spec):
            with open(os.environ["BUILD_LOG"], "a") as log:
                log.write(f"{os.getpid()}\\n")
            time.sleep(spec.seconds)
            self.result = np.arange(5)


    class Fragile(td.Record):
        __record_name__ = "Fragile"
        lease_ttl = 2.0  # seconds without a heartbeat before the lease is stale

        def build(self, spec):
            time.sleep(60)  # killed before it finishes
            self.result = np.arange(5)


    class Shared(td.Record):
        __record_name__ = "Shared"
        mutable = True
'''))
sys.path.insert(0, str(tmp))
from jobs import Fragile, Shared, Slow

env = {**os.environ, "TENSORDIR_ROOT": str(root), "BUILD_LOG": str(tmp / "builds.log"), "PYTHONPATH": str(tmp)}


def spawn(code):
    """Start a Python process that runs `code` with this notebook's root."""
    return subprocess.Popen([sys.executable, "-c", code], env=env, cwd=tmp)

## Six processes, one build

In [ ]:
racers = [spawn("from jobs import Slow; r = Slow.get(); assert r.status == 'done'") for _ in range(6)]
print("exit codes:", [p.wait() for p in racers])
print("builds:", len((tmp / "builds.log").read_text().splitlines()))
print(Slow.get().status, Slow.get().load("result"))

## Inspect a running build

While another process holds the lease, the record's status is `running`, and `owner.json` names
the builder (host, pid, batch job id from Slurm or LSF), with its last heartbeat and TTL.
`get(block=False)` raises `RecordBusy` instead of waiting; `get(timeout=...)` gives up with a
`TimeoutError`.

In [ ]:
builder = spawn("from jobs import Slow; Slow.get({'seconds': 3.0})")
record = Slow.locate({"seconds": 3.0})
lock = root / ".tensordir" / "locks" / "Slow" / record.key[:16]
while not (lock / "owner.json").exists():
    time.sleep(0.05)

owner = json.loads((lock / "owner.json").read_text())
print("status:", record.status, "| builder pid:", owner["pid"] == builder.pid, "| ttl:", owner["ttl"], "| job:", owner["job"])
print("temporary build:", [p.name for p in (root / ".tensordir" / "tmp").iterdir()])

try:
    Slow.get({"seconds": 3.0}, block=False)
except td.RecordBusy as exc:
    print("RecordBusy:", exc)
try:
    Slow.get({"seconds": 3.0}, timeout=0.5)
except TimeoutError as exc:
    print("TimeoutError:", exc)

finished = Slow.get({"seconds": 3.0})  # waits for the other process
print("after waiting:", finished.status, "| builder exit code:", builder.wait())

## A builder dies

`SIGKILL` gives the builder no chance to clean up. Its lease stays behind, so the record is
`running` until the heartbeat is older than `lease_ttl`, then `stale`. The next `get` breaks
the stale lease and builds again; here the class is patched so the rebuild finishes at once.

In [ ]:
victim = spawn("from jobs import Fragile; Fragile.get()")
fragile = Fragile.locate()
while fragile.status != "running":
    time.sleep(0.05)
os.kill(victim.pid, signal.SIGKILL)
victim.wait()

print("right after the kill:", fragile.status)
time.sleep(Fragile.lease_ttl + 0.5)
print("after the TTL:", fragile.status)


def quick_build(self, spec):
    self.result = [0, 1, 2]


Fragile.build = quick_build
print("rebuilt:", Fragile.get().status, Fragile.get().result)

Breaking the lease removed it, but the dead builder's temporary directory is still there.
`tensordir gc` removes stale leases, temporary builds without a live lease, and trash;
`--dry-run` lists them first.

In [ ]:
def tensordir(*args):
    result = subprocess.run([sys.executable, "-m", "tensordir.cli", *args], env=env, capture_output=True, text=True)
    print(result.stdout.replace(str(root), "<root>") or "(nothing to do)")


tensordir("gc", "--dry-run")
tensordir("gc")
tensordir("gc")

Two assumptions matter on clusters: node clocks must agree to well within the TTL (staleness
compares heartbeats written on different hosts), and the whole root, `.tensordir/` included,
must be on one filesystem so that `rename` is atomic. The lease only prevents duplicate work:
if two builders race after a lease is broken, the first rename wins and the second build is
discarded.

## Edits from many processes

Edits of a workspace record (`mutable = True`) each take the record's lease, re-read
`_record.json`, write one field and store the manifest atomically, so concurrent edits never
lose each other's fields.

In [ ]:
shared = Shared.get()
editors = [spawn(f"from jobs import Shared; Shared.get().write('worker_{i}', [{i}])") for i in range(8)]
print("exit codes:", [p.wait() for p in editors])
print(sorted(shared.keys()))

## Failures are records too

An exception in `build` stores a `failed` record with the traceback, which every process sees:
`get` raises `BuildFailed` until the record is deleted or rebuilt with `retry_failed=True`.
`KeyboardInterrupt` and `SystemExit` leave no record behind.

In [ ]:
code = "from jobs import Slow\nclass Broken(Slow):\n    __record_name__ = 'Broken'\n    def build(self, spec):\n        raise ValueError('bad input')\ntry:\n    Broken.get()\nexcept Exception as exc:\n    print(type(exc).__name__)"
print(subprocess.run([sys.executable, "-c", code], env=env, cwd=tmp, capture_output=True, text=True).stdout.strip())
tensordir("status")